# Ultraliser Neuron Skeletonization Notebook

Copyright (c) 2025 Open Brain Institute

Author(s): Marwan Abdellah < marwan.abdellah@openbraininstitute.org >

Last modified: 10.2025


## Imports and setting up platform authentication

Please follow the displayed instructions to authenticate.

In [ ]:
import os 
from obi_auth import get_token
from entitysdk.client import Client
from entitysdk.models import EMCellMesh
from entitysdk.models import EMDenseReconstructionDataset, BrainRegion

from ipywidgets import widgets
from neurom.check.runner import CheckRunner
from matplotlib import pyplot as plt
from obi_notebook import get_projects
from obi_notebook import get_entities

import ultraliser
ultraliser.copyrights()

## Selection of meshes and download

The meshes will be downloaded, then loaded and processed by Ultraliser.

#### Project selection
As a first step we select one of the projects we have access to that the meshes are associated with. 

In [ ]:
token = get_token(environment="staging", auth_mode="daf")
project_context = get_projects.get_projects(token, env="staging")

## Mesh selection

Next, we select the meshes. A widget with all the meshes with their IDs will appear. Simply select the mesh you are interested to skeletonize and proceed. Note that a single mesh can only be selected.

In [ ]:
client = Client(environment="staging", project_context=project_context, token_manager=token)
entity = client.search_entity(entity_type=EMCellMesh)

mesh_ids = []
entities = get_entities.get_entities(
    entity_type='em-cell-mesh', token=token, result=mesh_ids, project_context=project_context,
    multi_select=False, page_size=25, env='staging')

### Download the selected mesh

In [ ]:
# Since only one mesh is selected, we can directly fetch it
fetched = [client.get_entity(entity_id=mesh_id_, entity_type=EMCellMesh) for mesh_id_ in mesh_ids]

# Get the prefix 
prefix = os.path.splitext(os.path.basename(fetched[0].assets[0].path))[0]
output_path = f"{os.getcwd()}/{prefix}"
os.makedirs(output_path, exist_ok=True)

# Download the mesh asset to start processing it with Ultraliser
download_result = client.download_assets(fetched[0], output_path=output_path).all()

## Setup skeletonization parameters

In [ ]:
# Neuron skeletonization resolution (in microns)
neuron_voxel_size = 0.1  # in microns

# Spines skeletonization parameters
spines_voxel_size = 0.05  # in microns

# Set this flag to segment the spines or not
segment_spines = True

## Run the skeletonization process

In [ ]:
ultraliser.skeletonize_neuron_mesh(
    f"{output_path}/{fetched[0].assets[0].path}", output_path, neuron_voxel_size, spines_voxel_size, segment_spines)